# 🚀 Production-Ready LLM Fine-Tuning Pipeline (Colab Version)

This notebook contains the complete pipeline for fine-tuning an LLM using **QLoRA** (4-bit quantization) and **LoRA** adapters. 

### ⚠️ IMPORTANT: IF YOU GET AN OUT OF MEMORY (OOM) ERROR:
Go to the top menu and click **Runtime -> Restart session**, then run the cells again from the top. Colab holds onto memory from previous failed attempts!

### 🛠️ What's inside:
1. **Environment Setup**: Installing dependencies.
2. **Data Preparation**: Formatting the Dolly-15k dataset.
3. **Fine-Tuning**: Training an LLM with QLoRA. (Optimized to fit on a free T4 15GB GPU!)
4. **Evaluation**: Quantitative metrics and qualitative comparison.

## 1. Environment Setup
First, we install all the necessary libraries using the latest versions for maximum performance and stability.

In [ ]:
!pip install -q -U torch transformers peft datasets accelerate bitsandbytes wandb evaluate rouge-score bleu python-dotenv trl

## 2. Configuration & Tokens
Paste your tokens below. These are needed for Hugging Face (loading models) and WandB (tracking training).

*(Note: We are using `TinyLlama-1.1B` here because `Mistral-7B` requires more VRAM than the free Colab T4 GPU provides. This will train much faster and avoid OOM errors)*

In [ ]:
import os

# @title 🔑 Configuration
HF_TOKEN = "your_huggingface_token_here" # @param {type:"string"}
WANDB_API_KEY = "your_wandb_api_key_here" # @param {type:"string"}
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Small model for 15GB GPU constraints

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

import wandb
wandb.login(key=WANDB_API_KEY)

## 3. Data Preparation
We download the `databricks/databricks-dolly-15k` dataset and format it for instruction fine-tuning.

In [ ]:
import json
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def format_instruction(sample):
    instruction = sample.get("instruction", "")
    context = sample.get("context", "")
    response = sample.get("response", "")
    
    if context:
        text = f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n### Response:\n{response}"
    else:
        text = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Response:\n{response}"
    
    return {"text": text}

print("Downloading and formatting dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
formatted_dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

data_list = list(formatted_dataset)
train_data, val_data = train_test_split(data_list, test_size=0.1, random_state=42)

# Save locally for the trainer
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/train.json", "w") as f: json.dump(train_data, f)
with open("data/processed/validation.json", "w") as f: json.dump(val_data, f)

print(f"Setup complete! Train size: {len(train_data)}, Validation size: {len(val_data)}")

## 4. QLoRA Fine-Tuning
Loading the model in 4-bit and preparing the LoRA adapter. We explicitly clear memory to avoid PyTorch hoarding previous failed runs.

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# EXTREME MEMORY CLEANUP: Delete old variables if they exist in the globals to free RAM
for var in ['model', 'trainer', 'tokenizer']:
    if var in globals():
        del globals()[var]

# Clear cache from PyTorch GPU allocator
gc.collect()
torch.cuda.empty_cache()

# Validate GPU Memory State
if torch.cuda.is_available():
    print(f"Free GPU Memory before loading: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. BitsAndBytes Config (4-bit)
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

# 3. Load Model (Optimized for low VRAM)
print(f"\nLoading {BASE_MODEL_ID} into 4-bit memory...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False # Required for training
model.config.pretraining_tp = 1

# 4. Prepare for training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], # Lowered R values for memory savings
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 5. Load Processed Datasets safely handling the split
train_ds = load_dataset("json", data_files="data/processed/train.json", split="train")
eval_ds = load_dataset("json", data_files="data/processed/validation.json", split="train")

# 6. Modern SFTConfig for TRL >= 0.9.0
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type="constant",
    report_to="wandb",
    max_seq_length=256,
    dataset_text_field="text",
)

# 7. SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds.select(range(1000)), # Subset for faster Colab run
    eval_dataset=eval_ds.select(range(min(100, len(eval_ds)))), # Safe evaluation
    args=sft_config,
    peft_config=lora_config,
    tokenizer=tokenizer,
)

print("\nStarting training...")
trainer.train()

# 8. Save the Adapter
trainer.model.save_pretrained("./models/fine_tuned_adapter")
tokenizer.save_pretrained("./models/fine_tuned_adapter")
print("Model saved to ./models/fine_tuned_adapter")

## 5. Evaluation
Generate text and check the quality.

In [ ]:
from peft import PeftModel
import evaluate

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat are the benefits of LoRA?\n\n### Response:\n"
print("Generating sample response...")
print(generate(prompt))

# Calculate basic metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
print("Metrics libraries loaded successfully.")